In [1]:
import numpy as np
import pandas as pd

# 假数据：10 根 bar 的收盘价
close = pd.Series([10, 11, 10, 12, 13, 12, 14, 13, 15, 14], name="close")
ret = close.pct_change().fillna(0)
sma = close.rolling(3).mean()

# ---------- 1) 向量化回测 ----------
# 整列一次算信号；用 shift(1) 避免用到“当日收盘才知道”的信号去吃当日收益
signal_v = (close > sma).astype(int)          # 1=多头, 0=空仓
pnl_v = signal_v.shift(1).fillna(0) * ret
equity_v = (1 + pnl_v).cumprod()

# ---------- 2) 事件驱动回测 ----------
# 一根 bar 一根 bar 走；仓位是状态，只在穿越时买卖
cash, shares, position = 1.0, 0.0, 0
equity_e = []
for i in range(len(close)):
    price = close.iloc[i]
    # 信号在当根收盘后才“知道”，下一根才可交易 → 用上一根的 sma/close 决策更严谨
    # 这里为极简：用已算好的 signal_v.shift(1)，突出“循环 + 状态”而非向量算子
    want = int(signal_v.shift(1).fillna(0).iloc[i])

    if want == 1 and position == 0:           # 事件：开多
        shares = cash / price
        cash = 0.0
        position = 1
    elif want == 0 and position == 1:         # 事件：平仓
        cash = shares * price
        shares = 0.0
        position = 0

    equity_e.append(cash + shares * price)

equity_e = pd.Series(equity_e, index=close.index)

print(equity_v.round(4).tolist())
print(equity_e.round(4).tolist())

[1.0, 1.0, 1.0, 1.0, 1.0833, 1.0, 1.0, 0.9286, 0.9286, 0.8667]
[1.0, 1.0, 1.0, 1.0, 1.0, 0.9231, 1.0769, 1.0769, 1.2426, 1.2426]
